In [1]:
# Step 1: Clean slate — remove any conflicting pre-installed packages
!pip uninstall -y torch torchvision torchaudio dgl torchdata triton

Found existing installation: torch 2.2.2+cu121
Uninstalling torch-2.2.2+cu121:
  Successfully uninstalled torch-2.2.2+cu121
Found existing installation: torchvision 0.17.2+cu121
Uninstalling torchvision-0.17.2+cu121:
  Successfully uninstalled torchvision-0.17.2+cu121
Found existing installation: torchaudio 2.2.2+cu121
Uninstalling torchaudio-2.2.2+cu121:
  Successfully uninstalled torchaudio-2.2.2+cu121
Found existing installation: dgl 2.4.0+cu121
Uninstalling dgl-2.4.0+cu121:
  Successfully uninstalled dgl-2.4.0+cu121
Found existing installation: torchdata 0.7.1
Uninstalling torchdata-0.7.1:
  Successfully uninstalled torchdata-0.7.1
Found existing installation: triton 2.2.0
Uninstalling triton-2.2.0:
  Successfully uninstalled triton-2.2.0


In [2]:
# Step 2: Install PyTorch 2.2.2 + CUDA 12.1 (includes triton 2.2.0)
!pip install torch==2.2.2 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install torchdata==0.7.1
!pip install "numpy<2"  # torch 2.2.x was compiled against NumPy 1.x

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.2.2%2Bcu121-cp311-cp311-linux_x86_64.whl (757.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp311-cp311-linux_x86_64.whl (7.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp311-cp311-linux_x86_64.whl (3.4 MB)
  Using cached https://download-r2.pytorch.org/whl/triton-2.2.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (167.9 MB)
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.0%2Bcu121-cp311-cp311-linux_x86_64.whl (7.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.19.1%2Bcu121-cp311-cp311-linux_x86_64.whl (7.1 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121

In [3]:
# Step 3: Install DGL (must match torch 2.2 + CUDA 12.1)
!pip install dgl -f https://data.dgl.ai/wheels/torch-2.2/cu121/repo.html

Looking in links: https://data.dgl.ai/wheels/torch-2.2/cu121/repo.html
  Using cached https://data.dgl.ai/wheels/torch-2.2/cu121/dgl-2.4.0%2Bcu121-cp311-cp311-manylinux1_x86_64.whl (483.9 MB)

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [4]:
# Step 4: Install remaining dependencies
# transformers 4.35.2 is required for DNABERT-2 compatibility
!pip install transformers==4.35.2 accelerate datasets
!pip install cooler matplotlib biopython
!pip install "bitsandbytes<0.44" einops


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [5]:
# Step 5: Verify all versions are consistent
import torch
import dgl
import torchdata
import transformers
import numpy as np

print(f"Torch:        {torch.__version__}  (expect 2.2.2+cu121)")
print(f"CUDA:         {torch.cuda.is_available()}")
print(f"DGL:          {dgl.__version__}")
print(f"Torchdata:    {torchdata.__version__}")
print(f"Transformers: {transformers.__version__}  (expect 4.35.2)")
print(f"NumPy:        {np.__version__}  (expect < 2.0)")

/home/izpiz/coding/genome_paper/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/izpiz/coding/genome_paper/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


Torch:        2.2.2+cu121  (expect 2.2.2+cu121)
CUDA:         True
DGL:          2.4.0+cu121
Torchdata:    0.7.1
Transformers: 4.35.2  (expect 4.35.2)
NumPy:        1.26.4  (expect < 2.0)


In [6]:
# Step 6: Clear DNABERT-2 cache and patch flash_attn_triton.py
# The bundled triton flash-attention kernel uses the deprecated
# tl.dot(a, b, trans_b=True) API removed in Triton >= 2.1.
# We patch it to use tl.dot(a, tl.trans(b)) instead.
import shutil, os, glob

# Clear any stale cached modules so they get re-downloaded fresh
cache_dir = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules/zhihan1996")
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print("Cleared old DNABERT-2 module cache.")

# Patch flash_attn_triton.py everywhere it exists in the HF cache
# (hub snapshots AND modules/transformers_modules)
search_roots = [
    os.path.expanduser("~/.cache/huggingface/hub"),
    os.path.expanduser("~/.cache/huggingface/modules"),
]
patched = 0
for root in search_roots:
    for dirpath, dirnames, filenames in os.walk(root):
        if "flash_attn_triton.py" in filenames:
            fpath = os.path.join(dirpath, "flash_attn_triton.py")
            with open(fpath, "r") as f:
                content = f.read()
            if "trans_b=True" in content:
                content = content.replace(
                    "tl.dot(q, k, trans_b=True)",
                    "tl.dot(q, tl.trans(k))"
                )
                content = content.replace(
                    "tl.dot(do, v, trans_b=True)",
                    "tl.dot(do, tl.trans(v))"
                )
                with open(fpath, "w") as f:
                    f.write(content)
                print(f"Patched: {fpath}")
                patched += 1
            else:
                print(f"Already patched: {fpath}")

if patched == 0:
    print("No files needed patching (already clean).")
print("\nDone. If running in Jupyter, restart the kernel now before proceeding.")

Already patched: /home/izpiz/.cache/huggingface/hub/models--zhihan1996--DNABERT-2-117M/snapshots/7bce263b15377fc15361f52cfab88f8b586abda0/flash_attn_triton.py
No files needed patching (already clean).

Done. If running in Jupyter, restart the kernel now before proceeding.


In [10]:
# 1. Download Chromosome 22 FASTA file (compressed)
!wget https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr22.fa.gz

# 2. Unzip the file
!gunzip chr22.fa.gz

# 3. Rename it so it matches your Python code
!mv chr22.fa 1000genomes_reference.fasta

--2026-06-05 00:35:03--  https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr22.fa.gz
Resolving hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)|128.114.119.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12255678 (12M) [application/x-gzip]
Saving to: ‘chr22.fa.gz’

chr22.fa.gz         100%[===================>]  11.69M  2.30MB/s    in 7.3s    

2026-06-05 00:35:12 (1.61 MB/s) - ‘chr22.fa.gz’ saved [12255678/12255678]



In [11]:
# 1. Download the mcool file from NCBI GEO
# (The direct FTP URL is no longer available; using the GEO download gateway)
!wget "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM5057489&format=file&file=GSM5057489%5FU54%2DESC%2DFA%2DHindIII%2D20160311%2DR1%2DT1%5Fhg38%2Ehg38%2Emapq%5F30%2E1000%2Emcool" -O cell_line.mcool

--2026-06-05 00:35:36--  https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM5057489&format=file&file=GSM5057489%5FU54%2DESC%2DFA%2DHindIII%2D20160311%2DR1%2DT1%5Fhg38%2Ehg38%2Emapq%5F30%2E1000%2Emcool
Resolving www.ncbi.nlm.nih.gov (www.ncbi.nlm.nih.gov)... 2607:f220:41e:4290::110, 130.14.29.110
Connecting to www.ncbi.nlm.nih.gov (www.ncbi.nlm.nih.gov)|2607:f220:41e:4290::110|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1170977222 (1.1G) [application/octet-stream]
Saving to: ‘cell_line.mcool’

cell_line.mcool     100%[===================>]   1.09G  8.90MB/s    in 5m 10s  

2026-06-05 00:40:48 (3.60 MB/s) - ‘cell_line.mcool’ saved [1170977222/1170977222]



In [1]:
import os
os.environ["CPATH"] = "/usr/include/python3.12:" + os.environ.get("CPATH", "")

import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
from dgl.nn import GATConv
from transformers import AutoModel, AutoTokenizer
from transformers.models.bert.configuration_bert import BertConfig
import cooler
import numpy as np
from Bio import SeqIO
import time
import json
import bitsandbytes as bnb
from torch.cuda.amp import autocast, GradScaler
import torch.utils.checkpoint as checkpoint

# ==========================================
# 1. HYBRID MODEL ARCHITECTURE
# ==========================================
class DNABertGNN(nn.Module):
    def __init__(self, hidden_dim=768, num_classes=2, use_gradient_checkpointing=True):
        super().__init__()
        self.use_gradient_checkpointing = use_gradient_checkpointing
        
        # 1. Force the local BertConfig
        config = BertConfig.from_pretrained("zhihan1996/DNABERT-2-117M")
        
        # 2. Load the model (REMOVED the broken gradient_checkpointing_enable call)
        self.dnabert = AutoModel.from_pretrained(
            "zhihan1996/DNABERT-2-117M", 
            trust_remote_code=True, 
            config=config
        )
        self.bert_micro_batch = 4  # Process BERT in chunks of 4 to save VRAM
            
        # --- GAT Layers ---
        self.gat1 = GATConv(hidden_dim, hidden_dim // 4, num_heads=4, feat_drop=0.1, attn_drop=0.1)
        self.gat2 = GATConv(hidden_dim, hidden_dim, num_heads=1, feat_drop=0.1, attn_drop=0.1)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )
        
    def forward(self, input_ids, attention_mask, blocks):
        # --- FIX: Micro-batch DNABERT-2 to avoid OOM on small GPUs ---
        micro_batch = self.bert_micro_batch
        all_embeddings = []
        for mb_i in range(0, input_ids.size(0), micro_batch):
            ids_chunk = input_ids[mb_i:mb_i+micro_batch]
            mask_chunk = attention_mask[mb_i:mb_i+micro_batch]
            if self.use_gradient_checkpointing and self.training:
                def custom_forward(ids, mask):
                    return self.dnabert(input_ids=ids, attention_mask=mask)[0]
                out = checkpoint.checkpoint(custom_forward, ids_chunk, mask_chunk, use_reentrant=False)
            else:
                out = self.dnabert(input_ids=ids_chunk, attention_mask=mask_chunk)[0]
            all_embeddings.append(out.mean(dim=1))
        seq_embeddings = torch.cat(all_embeddings, dim=0)

        # Structural Encoding
        h = seq_embeddings 
        h = self.gat1(blocks[0], h).flatten(1)
        h = F.elu(h)
        h = self.gat2(blocks[1], h).squeeze(1)
        
        # Modality Fusion
        fused_embeddings = torch.cat([seq_embeddings, h], dim=1)
        
        # Classification
        return self.classifier(fused_embeddings)
# ==========================================
# 2. DATA PIPELINE
# ==========================================
def build_biological_graph(hic_cool_path, fasta_path, resolution=50000, max_seq_len=4000):
    print(f"Loading Hi-C map from {hic_cool_path} at {resolution}bp resolution...")
    c = cooler.Cooler(f'{hic_cool_path}::/resolutions/{resolution}')
    
    matrix = c.matrix(balance=False, sparse=True)[:]
    src, dst = matrix.nonzero()
    weights = matrix.data
    
    g = dgl.graph((src, dst))
    g.edata['weight'] = torch.tensor(weights, dtype=torch.float32)
    g = dgl.add_self_loop(g)
    
    bins = c.bins()[:]
    print(f"Graph initialized with {g.num_nodes()} nodes. Mapping FASTA sequences...")
    
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)
    genome_records = SeqIO.to_dict(SeqIO.parse(fasta_path, "fasta"))
    
    input_ids_list = []
    attention_mask_list = []
    
    for idx, row in bins.iterrows():
        chrom = row['chrom']
        start = row['start']
        end = row['end']
        
        center = (start + end) // 2
        window_start = max(0, center - (max_seq_len // 2))
        window_end = center + (max_seq_len // 2)
        
        if chrom in genome_records:
            seq = str(genome_records[chrom].seq[window_start:window_end]).upper()
        else:
            seq = "N" * max_seq_len
            
        encoded = tokenizer(
            seq, padding='max_length', truncation=True, max_length=512, return_tensors="pt"
        )
        
        input_ids_list.append(encoded['input_ids'].squeeze(0))
        attention_mask_list.append(encoded['attention_mask'].squeeze(0))
        
        if (idx + 1) % 5000 == 0:
            print(f"Processed {idx + 1} / {len(bins)} genomic bins...")

    g.ndata['input_ids'] = torch.stack(input_ids_list)
    g.ndata['attention_mask'] = torch.stack(attention_mask_list)
    g.ndata['label'] = torch.randint(0, 2, (g.num_nodes(),))
    
    print("Biological Graph Construction Complete.")
    return g

def create_biological_sampler(g, batch_size=8, fanout=None):
    if fanout is None:
        fanout = [4, 4]
    sampler = dgl.dataloading.MultiLayerNeighborSampler(fanout) 
    train_nids = torch.arange(g.num_nodes())
    
    dataloader = dgl.dataloading.DataLoader(
        g, train_nids, sampler,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        num_workers=0 # CRITICAL: Kept at 0 for Jupyter Notebook compatibility
    )
    return dataloader

# ==========================================
# 3. BENCHMARKING ENGINE
# ==========================================
def run_biological_benchmark(dataloader, device_name, vram_limit_gb):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

    print(f"\n--- Commencing Biological Run on {device_name} ({vram_limit_gb}GB Profile) ---")
    
    model = DNABertGNN(use_gradient_checkpointing=True).to(device)
    
    if vram_limit_gb <= 8:
        print("Engaging 8-bit AdamW for extreme 6GB/8GB VRAM conservation.")
        optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=2e-5)
        accum_steps = 8 
    else:
        print("Engaging 32-bit AdamW for standard 16GB VRAM (Tesla T4).")
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
        accum_steps = 4
        
    scaler = GradScaler()
    metrics = {'epoch': [], 'loss': [], 'peak_vram_mb': [], 'throughput_it_s': []}

    model.train()
    for epoch in range(5): 
        epoch_loss = 0
        start_time = time.time()
        
        for step, (input_nodes, output_nodes, blocks) in enumerate(dataloader):
            input_ids = blocks[0].srcdata['input_ids'].to(device)
            attn_mask = blocks[0].srcdata['attention_mask'].to(device)
            labels = blocks[-1].dstdata['label'].to(device)
            
            blocks = [b.to(device) for b in blocks]

            with autocast(dtype=torch.float16):
                logits = model(input_ids, attn_mask, blocks)
                loss = torch.nn.functional.cross_entropy(logits, labels)
                loss = loss / accum_steps 

            scaler.scale(loss).backward()

            if (step + 1) % accum_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            epoch_loss += loss.item() * accum_steps
            
            if step == 50: break # Benchmark limit per epoch

        end_time = time.time()
        peak_vram = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
        throughput = 50 / (end_time - start_time)
        
        metrics['epoch'].append(epoch + 1)
        metrics['loss'].append(epoch_loss / 50)
        metrics['peak_vram_mb'].append(peak_vram)
        metrics['throughput_it_s'].append(throughput)
        
        print(f"Epoch {epoch+1} | Loss: {epoch_loss/50:.4f} | VRAM: {peak_vram:.1f} MB | Throughput: {throughput:.2f} it/s")

    with open(f'benchmark_{device_name}.json', 'w') as f:
        json.dump(metrics, f)
        
    return metrics

# ==========================================
# 4. EXECUTION BLOCK
# ==========================================
# IMPORTANT: Update these to point to your actual downloaded files!
HIC_FILE = "cell_line.mcool" 
FASTA_FILE = "1000genomes_reference.fasta"

# 1. Build Data
graph = build_biological_graph(HIC_FILE, FASTA_FILE, resolution=50000)

# 2. Run Benchmark (Uncomment the ONE you are currently testing)
# --- RTX 3050 6GB: batch_size=2, fanout=[2,2] to fit in VRAM ---
dataloader = create_biological_sampler(graph, batch_size=2, fanout=[2, 2])
run_biological_benchmark(dataloader, "RTX3050", vram_limit_gb=6)
# --- RTX 4060 8GB ---
# dataloader = create_biological_sampler(graph, batch_size=4, fanout=[3, 3])
# run_biological_benchmark(dataloader, "RTX4060", vram_limit_gb=8)
# --- Tesla T4 16GB ---
# dataloader = create_biological_sampler(graph, batch_size=8, fanout=[4, 4])
# run_biological_benchmark(dataloader, "T4", vram_limit_gb=16)

/home/izpiz/coding/genome_paper/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/izpiz/coding/genome_paper/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


Loading Hi-C map from cell_line.mcool at 50000bp resolution...
Graph initialized with 61776 nodes. Mapping FASTA sequences...


/home/izpiz/coding/genome_paper/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Processed 5000 / 61776 genomic bins...
Processed 10000 / 61776 genomic bins...
Processed 15000 / 61776 genomic bins...
Processed 20000 / 61776 genomic bins...
Processed 25000 / 61776 genomic bins...
Processed 30000 / 61776 genomic bins...
Processed 35000 / 61776 genomic bins...
Processed 40000 / 61776 genomic bins...
Processed 45000 / 61776 genomic bins...
Processed 50000 / 61776 genomic bins...
Processed 55000 / 61776 genomic bins...
Processed 60000 / 61776 genomic bins...
Biological Graph Construction Complete.

--- Commencing Biological Run on RTX3050 (6GB Profile) ---


/home/izpiz/coding/genome_paper/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/izpiz/coding/genome_paper/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/izpiz/coding/genome_paper/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.den

Engaging 8-bit AdamW for extreme 6GB/8GB VRAM conservation.


/home/izpiz/coding/genome_paper/.venv/lib/python3.11/site-packages/dgl/dataloading/dataloader.py:1144: DGLWarning: Dataloader CPU affinity opt is not enabled, consider switching it on (see enable_cpu_affinity() or CPU best practices for DGL [https://docs.dgl.ai/tutorials/cpu/cpu_best_practises.html])
  dgl_warning(


: 